In [1]:
"""
=============================================================================
CMPRSR FOLLOW-UP: TWO-SIDED COMPRESSION CONTROL
=============================================================================
Research question
-----------------
Cmprsr's length reward (arXiv:2511.12281, eq. 1) is

    R_len = 1 - max(0, r_C - r_T),      r_C = compressed_tokens / original_tokens

It penalises only r_C > r_T (output LONGER than target). For every r_C < r_T
(output SHORTER than target) it is pinned at 1.0 — a flat region with no
length gradient. At r_T = 0.3, compressing to 0.30 and to 0.05 earn the
identical length reward.

Meanwhile standard GRPO normalises the per-sequence loss by 1/|o_i|, so
SHORTER outputs receive LARGER per-token gradient weight. The optimiser
therefore amplifies exactly the region the reward has stopped policing.

Hypothesis: adding an explicit short-side penalty tightens the Delta_CR
distribution without degrading semantic retention.

Terminology note
----------------
The paper calls r_C > r_T "over-compression". Under the standard reading of
CR = compressed/original, r_C > r_T is a LONGER output. This script always
states the inequality explicitly and never relies on the label.

Design decisions (differ from a naive implementation — read before citing)
-------------------------------------------------------------------------
1. The proposed reward is ADDITIVE, not symmetric:

       R_len_two = 1 - max(0, r_C - r_T) - lam * max(0, r_T - r_C)

   For r_C >= r_T this is bit-identical to eq. 1. Only the short side
   changes. A symmetric reward (1 - lam*|r_C - r_T|) would also steepen the
   LONG side, confounding the ablation: a tighter Delta_CR could then come
   from the harsher long slope rather than from policing the flat region.
   With this form, `lam` is the single manipulated variable.

2. The reward is NOT clipped at zero. Clipping recreates a flat region at
   the bottom (e.g. lam=4, r_T=0.7: everything below r_C=0.45 returns 0.0,
   identical reward, no gradient) — reintroducing the exact pathology under
   study. GRPO uses group-relative advantages, so negative absolute rewards
   are harmless. An assertion enforces strict monotonicity on [0, r_T].

3. Both arms train with STANDARD GRPO (1/L normalisation intact). The
   reward is the only difference. A Dr. GRPO arm is a separate ablation and
   is left as future work so this experiment stays single-variable.

4. R_qual is number retention, not answer exact-match. See CELL 12.

Hardware: single NVIDIA T4 (15 GB) — Kaggle free tier. 4-bit QLoRA.
=============================================================================
"""

import subprocess, sys, os

# =============================================================================
# CELL 1 — Environment diagnostics
# =============================================================================

def cell1_diagnostics():
    print("=" * 70)
    print("CELL 1: Environment Diagnostics")
    print("=" * 70)
    try:
        import torch
        print(f"PyTorch version     : {torch.__version__}")
        if torch.cuda.is_available():
            gpu = torch.cuda.get_device_properties(0)
            total_vram = gpu.total_memory / 1024**3
            print(f"GPU name            : {gpu.name}")
            print(f"Total VRAM          : {total_vram:.2f} GB")
            print(f"CUDA version        : {torch.version.cuda}")
            if total_vram < 14.0:
                print(f"WARNING: only {total_vram:.1f} GB VRAM — safe mode will engage.")
            else:
                print("GPU check: OK (>=14 GB VRAM)")
        else:
            print("WARNING: No CUDA GPU. Phase 1 will run; Phase 2 will be skipped.")
    except ImportError:
        print("PyTorch not yet installed.")
    for mod in ["transformers", "peft", "trl", "bitsandbytes"]:
        try:
            m = __import__(mod)
            print(f"{mod:20s}: {getattr(m, '__version__', 'unknown')}")
        except ImportError:
            print(f"{mod:20s}: not installed yet")
    print("=" * 70)

cell1_diagnostics()


# =============================================================================
# CELL 2 — Install dependencies
# =============================================================================

def cell2_install():
    print("\n" + "=" * 70)
    print("CELL 2: Installing dependencies")
    print("=" * 70)
    packages = [
        "transformers>=4.45.0", "peft>=0.12.0", "bitsandbytes>=0.43.0",
        "datasets>=2.20.0", "accelerate>=0.33.0", "scipy>=1.11.0",
        "sentencepiece>=0.1.99", "protobuf>=3.20.0",
    ]
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet"] + packages,
        capture_output=True, text=True)
    if result.returncode != 0:
        print("pip stderr:", result.stderr[-2000:])
        raise RuntimeError("Dependency installation failed.")
    print("Installation complete.")

cell2_install()


# =============================================================================
# CELL 3 — Imports
# =============================================================================

print("\n" + "=" * 70)
print("CELL 3: Imports")
print("=" * 70)

import gc, json, math, re, time, warnings, traceback, random
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass, field, asdict
from contextlib import contextmanager

import numpy as np
import torch
import torch.nn.functional as F

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    GenerationConfig, get_linear_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
print("All imports successful.")


# =============================================================================
# CELL 4 — Reproducibility + memory utilities
# =============================================================================

print("\n" + "=" * 70)
print("CELL 4: Reproducibility + Memory Utilities")
print("=" * 70)

SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Seeds set to {seed}")

set_seed(SEED)

def get_gpu_memory() -> Dict[str, float]:
    if not torch.cuda.is_available():
        return {"allocated_gb": 0.0, "reserved_gb": 0.0, "total_gb": 0.0}
    return {
        "allocated_gb": torch.cuda.memory_allocated(0) / 1024**3,
        "reserved_gb": torch.cuda.memory_reserved(0) / 1024**3,
        "total_gb": torch.cuda.get_device_properties(0).total_memory / 1024**3,
    }

def log_memory(label: str = ""):
    m = get_gpu_memory()
    print(f"[MEM {label}] alloc={m['allocated_gb']:.2f}GB "
          f"reserved={m['reserved_gb']:.2f}GB total={m['total_gb']:.2f}GB")
    return m

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.synchronize()

def get_available_vram_gb() -> float:
    if not torch.cuda.is_available():
        return 0.0
    p = torch.cuda.get_device_properties(0)
    return p.total_memory / 1024**3 - torch.cuda.memory_allocated(0) / 1024**3

@contextmanager
def memory_scope(label: str):
    before = get_gpu_memory()
    print(f"[SCOPE START: {label}] alloc={before['allocated_gb']:.2f}GB")
    try:
        yield
    finally:
        after = get_gpu_memory()
        print(f"[SCOPE END:   {label}] alloc={after['allocated_gb']:.2f}GB "
              f"(delta={after['allocated_gb'] - before['allocated_gb']:+.2f}GB)")

OUTPUT_DIR = Path("/kaggle/working/cmprsr_research")
if not Path("/kaggle").exists():
    OUTPUT_DIR = Path("./cmprsr_research")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR = OUTPUT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")
log_memory("after cell 4")


# =============================================================================
# CELL 5 — Configuration
# =============================================================================

print("\n" + "=" * 70)
print("CELL 5: Configuration")
print("=" * 70)

@dataclass
class ExperimentConfig:
    seed: int = SEED

    # --- Model ---
    model_name: str = "Qwen/Qwen3-4B-Instruct-2507"
    model_name_fallback: str = "Qwen/Qwen3-4B"

    # --- Quantisation ---
    load_in_4bit: bool = True
    bnb_4bit_quant_type: str = "nf4"
    bnb_4bit_double_quant: bool = True

    # --- LoRA ---
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_target_modules: List[str] = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"])

    # --- Sequence budget ---
    max_input_tokens: int = 256
    max_new_tokens: int = 192
    max_seq_len: int = 512

    # --- Training ---
    grad_accum_steps: int = 8
    group_size: int = 4            # G; paper uses G=4 (Appendix D.1)
    learning_rate: float = 5e-5
    num_steps: int = 60
    warmup_steps: int = 5
    max_grad_norm: float = 1.0
    weight_decay: float = 0.01
    gen_temperature: float = 0.9   # rollout sampling temperature
    gen_top_p: float = 0.95

    # --- Dataset ---
    dataset_name: str = "openai/gsm8k"
    dataset_config: str = "main"
    train_size: int = 80
    eval_size: int = 30
    debug_size: int = 8

    # --- Target ratio sampling (matches paper: r_T ~ U(0.1, 0.7)) ---
    r_T_min: float = 0.1
    r_T_max: float = 0.7

    # --- Reward ---
    lambda_two_sided: float = 2.0
    lambda_strong: float = 4.0
    kl_coeff: float = 0.01
    epsilon_denom: float = 1e-8
    use_solver_qual: bool = False   # see CELL 12; costs 1 extra generation/rollout

    # --- Phases ---
    run_phase1: bool = True
    run_phase2: bool = True
    safe_mode: bool = False

CFG = ExperimentConfig()

available = get_available_vram_gb()
if torch.cuda.is_available() and available < 10.0:
    CFG.safe_mode = True
    CFG.max_input_tokens, CFG.max_new_tokens, CFG.max_seq_len = 192, 128, 384
    CFG.train_size, CFG.eval_size, CFG.num_steps = 40, 20, 30
    CFG.group_size = 2
    print(f"SAFE MODE ACTIVATED (available VRAM {available:.1f}GB).")
if not torch.cuda.is_available():
    CFG.run_phase2 = False
    print("No GPU: Phase 2 disabled, Phase 1 (analytical) will still run.")

print("\nFinal configuration:")
for k, v in asdict(CFG).items():
    print(f"  {k:26s}: {v}")
with open(OUTPUT_DIR / "config.json", "w") as f:
    json.dump(asdict(CFG), f, indent=2)
log_memory("after config")


# =============================================================================
# CELL 12a — Reward implementation  (defined early: Phase 1 needs it)
# =============================================================================

def r_len_original(r_C: float, r_T: float) -> float:
    """Cmprsr eq. (1). Flat at 1.0 for all r_C <= r_T."""
    return 1.0 - max(0.0, r_C - r_T)

def r_len_original_batch(r_C: np.ndarray, r_T: float) -> np.ndarray:
    return 1.0 - np.maximum(0.0, r_C - r_T)

def r_len_two_sided(r_C: float, r_T: float, lam: float = 2.0) -> float:
    """
    ADDITIVE two-sided reward.

        R = 1 - max(0, r_C - r_T) - lam * max(0, r_T - r_C)

    For r_C >= r_T this is IDENTICAL to eq. (1): the long side is untouched,
    so `lam` is the only manipulated variable in the ablation.
    Deliberately NOT clipped at 0 — clipping would recreate a flat, zero-
    gradient region on the short side, which is the pathology under study.
    """
    return 1.0 - max(0.0, r_C - r_T) - lam * max(0.0, r_T - r_C)

def r_len_two_sided_batch(r_C: np.ndarray, r_T: float, lam: float = 2.0) -> np.ndarray:
    return (1.0 - np.maximum(0.0, r_C - r_T)
            - lam * np.maximum(0.0, r_T - r_C))

def delta_cr(r_C: float, r_T: float) -> float:
    """Paper's definition. Negative => output shorter than target."""
    return r_C - r_T

def compute_compression_ratio(orig_tokens: int, comp_tokens: int,
                              eps: float = 1e-8) -> float:
    """r_C = compressed / original.  NOT inverted."""
    if orig_tokens <= 0:
        return 0.0
    return comp_tokens / (orig_tokens + eps)


# ---- assertions -------------------------------------------------------------
print("\n" + "=" * 70)
print("Reward assertions")
print("=" * 70)
_rT = 0.30
assert abs(r_len_original(0.30, _rT) - 1.0) < 1e-9
assert abs(r_len_original(0.05, _rT) - 1.0) < 1e-9
print(f"  R_len_orig(0.30, 0.30) = {r_len_original(0.30,_rT):.4f}")
print(f"  R_len_orig(0.05, 0.30) = {r_len_original(0.05,_rT):.4f}")
print("  -> identical: flat region confirmed")
assert abs(r_len_original(0.50, _rT) - 0.80) < 1e-9
print(f"  R_len_orig(0.50, 0.30) = {r_len_original(0.50,_rT):.4f}  (penalised)")

# long side must be untouched by the proposal
for _rc in [0.30, 0.35, 0.5, 0.7, 1.0]:
    assert abs(r_len_two_sided(_rc, _rT, 2.0) - r_len_original(_rc, _rT)) < 1e-9, \
        "FAIL: proposed reward altered the long side — ablation confounded!"
print("  Long side (r_C >= r_T) identical to eq.(1) for lam=2  [OK]")

# short side must be strictly monotone (no flat/zero-gradient region)
_grid = np.linspace(0.0, _rT, 200)
for lam in [2.0, 4.0]:
    _vals = r_len_two_sided_batch(_grid, _rT, lam)
    assert np.all(np.diff(_vals) > 0), f"FAIL: flat region on short side at lam={lam}"
print("  Short side strictly increasing for lam in {2,4} — no flat region  [OK]")
print(f"  R_len_two(0.05, 0.30, lam=2) = {r_len_two_sided(0.05,_rT,2.0):.4f}")
print(f"  R_len_two(0.00, 0.30, lam=4) = {r_len_two_sided(0.00,_rT,4.0):.4f} "
      f"(negative is fine: GRPO advantages are group-relative)")
assert abs(compute_compression_ratio(1000, 50) - 0.05) < 1e-4
assert compute_compression_ratio(1000, 50) < _rT
print("  Sign check: r_C=0.05 < r_T=0.30 => output shorter than target  [OK]")
print("\nAll reward assertions PASSED")


# =============================================================================
# CELL 13 — Analytical reward audit (Phase 1)
# =============================================================================

print("\n" + "=" * 70)
print("CELL 13: Analytical Reward Audit  [ANALYTICAL — not a training result]")
print("=" * 70)

R_C_GRID = np.linspace(0.0, 1.0, 1000)
TARGET_RATIOS_AUDIT = [0.1, 0.2, 0.3, 0.5, 0.7]

print("\nTable 1: flat-region width of the original reward")
print(f"{'r_T':>6} {'flat_width':>12} {'R@r_C=0':>10} {'R@r_C=r_T':>11} {'R@r_C=1':>10}")
print("-" * 52)
for r_T in TARGET_RATIOS_AUDIT:
    print(f"{r_T:>6.2f} {r_T:>12.3f} {r_len_original(0.0,r_T):>10.4f} "
          f"{r_len_original(r_T,r_T):>11.4f} {r_len_original(1.0,r_T):>10.4f}")

print("\nTable 2: original vs additive two-sided (r_T = 0.30)")
print(f"{'r_C':>8} {'R_orig':>10} {'R_two(2)':>10} {'R_two(4)':>10} "
      f"{'orig_pen':>10} {'two_pen':>9}")
print("-" * 62)
for r_C in [0.0, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.70, 1.0]:
    o = r_len_original(r_C, 0.30)
    t2 = r_len_two_sided(r_C, 0.30, 2.0)
    t4 = r_len_two_sided(r_C, 0.30, 4.0)
    print(f"{r_C:>8.3f} {o:>10.4f} {t2:>10.4f} {t4:>10.4f} "
          f"{'YES' if o < 1.0 else 'flat':>10} {'YES' if t2 < 1.0 else 'flat':>9}")

AUDIT_DATA = {
    "r_C_grid": R_C_GRID,
    "target_ratios": TARGET_RATIOS_AUDIT,
    "original_rewards": {t: r_len_original_batch(R_C_GRID, t) for t in TARGET_RATIOS_AUDIT},
    "two_lam2": {t: r_len_two_sided_batch(R_C_GRID, t, 2.0) for t in TARGET_RATIOS_AUDIT},
    "two_lam4": {t: r_len_two_sided_batch(R_C_GRID, t, 4.0) for t in TARGET_RATIOS_AUDIT},
}


# =============================================================================
# CELL 14 — GRPO weighting simulation (Phase 1)
# =============================================================================

print("\n" + "=" * 70)
print("CELL 14: GRPO Per-Token Weighting  [ANALYTICAL]")
print("=" * 70)
print("""
Standard GRPO normalises the per-sequence loss by 1/|o_i|. For a fixed
sequence-level advantage, per-token gradient weight is proportional to 1/L:
shorter outputs get MORE weight per token. Dr. GRPO removes this term.
""")

L_GRID = np.arange(10, 301, dtype=float)
TYPICAL_L_IN = 80.0
r_T_sim = 0.30
L_target = r_T_sim * TYPICAL_L_IN          # output length that hits target

# Scale-free anchoring: express weight relative to the weight AT TARGET LENGTH.
# This is invariant to the chosen sweep range (unlike anchoring at L=100).
grpo_rel = (1.0 / L_GRID) / (1.0 / L_target)
drgrpo_rel = np.ones_like(L_GRID)
r_C_from_L = L_GRID / TYPICAL_L_IN
flat_mask = r_C_from_L <= r_T_sim

print(f"Anchor: target length L* = r_T * L_in = {L_target:.0f} tokens "
      f"(weight defined as 1.0x there)\n")
print(f"{'fraction of target length':>26} {'r_C':>8} {'rel. per-token weight':>22}")
print("-" * 60)
for frac in [0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 4.0]:
    L = frac * L_target
    print(f"{frac:>25.2f}x {L/TYPICAL_L_IN:>8.3f} {(1.0/L)/(1.0/L_target):>21.2f}x")

print(f"\n  HEADLINE (scale-free): an output at HALF the target length receives "
      f"{2.0:.0f}x the per-token gradient weight of one at target;")
print(f"  at a quarter of target length, {4.0:.0f}x. Both sit inside the flat "
      f"region where R_len provides no length gradient.")
print("  Dr. GRPO would equalise all of these to 1.0x.")

WEIGHTING_DATA = {
    "L_grid": L_GRID, "r_C_from_L": r_C_from_L,
    "grpo_weight": grpo_rel, "dr_grpo_weight": drgrpo_rel,
    "flat_mask": flat_mask, "r_T": r_T_sim,
    "typical_L_in": TYPICAL_L_IN, "L_target": L_target,
}


# =============================================================================
# CELL 15 — Figures 1–3
# =============================================================================

print("\n" + "=" * 70)
print("CELL 15: Figures 1-3  [ANALYTICAL]")
print("=" * 70)

plt.rcParams.update({
    "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
    "legend.fontsize": 9, "figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.3,
})

# ---- Figure 1 ---------------------------------------------------------------
fig1, ax1 = plt.subplots(figsize=(10, 5.5))
r_T_fig, x = 0.30, AUDIT_DATA["r_C_grid"]
ax1.axvspan(0.0, r_T_fig, alpha=0.12, color="red",
            label=f"$r_C < r_T$ = {r_T_fig}  (original $R_{{len}}$ = 1.0 here)")
ax1.plot(x, AUDIT_DATA["original_rewards"][r_T_fig], "b-", lw=2.8,
         label="Original $R_{len}$ (Cmprsr eq. 1)")
ax1.plot(x, AUDIT_DATA["two_lam2"][r_T_fig], "g--", lw=2.2,
         label=r"Proposed additive two-sided ($\lambda$=2)")
ax1.plot(x, AUDIT_DATA["two_lam4"][r_T_fig], "m:", lw=2.2,
         label=r"Proposed additive two-sided ($\lambda$=4)")
ax1.axvline(r_T_fig, color="k", ls="--", lw=1.2, alpha=0.6)
ax1.axhline(0.0, color="grey", lw=0.8, alpha=0.5)
ax1.annotate("$r_C$=0.05 and $r_C$=0.30\nboth receive $R_{len}$=1.0",
             xy=(0.05, 1.0), xytext=(0.09, 0.55),
             arrowprops=dict(arrowstyle="->", color="red", lw=1.5),
             color="red", fontsize=9,
             bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                       edgecolor="red", alpha=0.9))
ax1.text(0.62, 0.72,
         "Long side ($r_C > r_T$) is IDENTICAL\nfor all three curves —\n"
         "only the short side changes",
         fontsize=8.5, color="darkgreen", ha="left",
         bbox=dict(boxstyle="round", facecolor="honeydew", alpha=0.85))
ax1.set_xlabel(r"Achieved ratio $r_C$ = compressed_tokens / original_tokens")
ax1.set_ylabel(r"$R_{len}$")
ax1.set_title("Figure 1 [Analytical]: Original vs. Additive Two-Sided Length Reward\n"
              f"$r_T$={r_T_fig} — mechanism audit, not a training result")
ax1.set_xlim(-0.02, 1.02); ax1.set_ylim(-1.35, 1.2)
ax1.legend(loc="lower right")
fig1.tight_layout()
fig1.savefig(FIGURES_DIR / "figure1_reward_audit.png", bbox_inches="tight")
plt.close(fig1); print("Figure 1 saved.")

# ---- Figure 2 ---------------------------------------------------------------
fig2, axes2 = plt.subplots(1, 2, figsize=(12.5, 5))

ax = axes2[0]
L = WEIGHTING_DATA["L_grid"]
ax.axvspan(L.min(), WEIGHTING_DATA["L_target"], alpha=0.12, color="red",
           label=f"flat-$R_{{len}}$ region ($L < {WEIGHTING_DATA['L_target']:.0f}$)")
ax.plot(L, WEIGHTING_DATA["grpo_weight"], "b-", lw=2.2,
        label="Standard GRPO ($\\propto 1/L$)")
ax.plot(L, WEIGHTING_DATA["dr_grpo_weight"], "g--", lw=2.2, label="Dr. GRPO (flat)")
ax.axvline(WEIGHTING_DATA["L_target"], color="k", ls=":", lw=1.2)
ax.axhline(1.0, color="k", ls=":", alpha=0.4)
ax.annotate("shorter outputs receive\nlarger per-token weight",
            xy=(18, (1.0/18)/(1.0/WEIGHTING_DATA["L_target"])), xytext=(80, 3.6),
            arrowprops=dict(arrowstyle="->", color="blue"), color="blue", fontsize=9)
ax.set_xlabel("Output length (tokens)")
ax.set_ylabel(f"Per-token weight, relative to target length "
              f"L*={WEIGHTING_DATA['L_target']:.0f}")
ax.set_title("GRPO per-token weighting vs. output length\n[Analytical]")
ax.set_ylim(0, 5); ax.legend(fontsize=8, loc="upper right")

ax = axes2[1]
rc, w = WEIGHTING_DATA["r_C_from_L"], WEIGHTING_DATA["grpo_weight"]
ax.axvspan(0.0, WEIGHTING_DATA["r_T"], alpha=0.12, color="red")
l1, = ax.plot(rc, r_len_original_batch(rc, WEIGHTING_DATA["r_T"]), "b-", lw=2.2,
              label="Original $R_{len}$")
ax.set_ylim(0.0, 1.12)                      # FIX: was autoscaling to -2.5
ax.set_ylabel("$R_{len}$", color="blue")
ax.tick_params(axis="y", labelcolor="blue")
ax2r = ax.twinx()
l2, = ax2r.plot(rc, w, color="darkorange", lw=2.2, ls="--",
                label="GRPO per-token weight")
ax2r.set_ylim(0, 5)
ax2r.set_ylabel("Relative GRPO weight", color="darkorange")
ax2r.tick_params(axis="y", labelcolor="darkorange")
ax.text(0.52, 0.30,
        "Flat region: $R_{len}$ is constant\nbut per-token weight is HIGHEST",
        fontsize=9, color="darkred",
        bbox=dict(boxstyle="round", facecolor="mistyrose", alpha=0.85))
ax.set_xlabel("$r_C$"); ax.set_xlim(0, 1)
ax.set_title("Flat $R_{len}$ region coincides with\nhighest per-token weight [Analytical]")
ax.legend([l1, l2], [l1.get_label(), l2.get_label()], fontsize=8, loc="center right")

fig2.suptitle("Figure 2 [Analytical]: GRPO Weighting Geometry", fontsize=12.5)
fig2.tight_layout()
fig2.savefig(FIGURES_DIR / "figure2_grpo_weighting.png", bbox_inches="tight")
plt.close(fig2); print("Figure 2 saved.")

# ---- Figure 3 ---------------------------------------------------------------
fig3, axes3 = plt.subplots(1, 2, figsize=(14, 5))
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(TARGET_RATIOS_AUDIT)))
for i, r_T in enumerate(TARGET_RATIOS_AUDIT):
    axes3[0].plot(x, AUDIT_DATA["original_rewards"][r_T], color=colors[i], lw=2,
                  label=f"$r_T$={r_T}")
    axes3[1].plot(x, AUDIT_DATA["two_lam2"][r_T], color=colors[i], lw=2,
                  label=f"$r_T$={r_T}")
    axes3[0].axvline(r_T, color=colors[i], ls=":", alpha=0.35, lw=0.9)
    axes3[1].axvline(r_T, color=colors[i], ls=":", alpha=0.35, lw=0.9)
axes3[0].set_title("Original $R_{len}$: flat region widens with $r_T$ [Analytical]")
axes3[1].set_title(r"Additive two-sided ($\lambda$=2): no flat region at any $r_T$")
axes3[0].set_ylim(-0.05, 1.15)
axes3[1].set_ylim(-1.5, 1.15)
for ax in axes3:
    ax.set_xlabel("$r_C$"); ax.set_ylabel("$R_{len}$")
    ax.set_xlim(0, 1); ax.axhline(0.0, color="grey", lw=0.8, alpha=0.5)
    ax.legend(fontsize=8)
fig3.suptitle("Figure 3 [Analytical]: Target-Ratio Sensitivity", fontsize=12.5)

print("\nFigure 3 summary — the flat region is exactly r_T wide:")
print(f"{'r_T':>6} {'flat_width':>12} {'orig R@0':>10} {'two R@0 (lam=2)':>17}")
print("-" * 48)
for r_T in TARGET_RATIOS_AUDIT:
    print(f"{r_T:>6.2f} {r_T:>12.3f} {r_len_original(0.0,r_T):>10.4f} "
          f"{r_len_two_sided(0.0,r_T,2.0):>17.4f}")

fig3.tight_layout()
fig3.savefig(FIGURES_DIR / "figure3_target_sensitivity.png", bbox_inches="tight")
plt.close(fig3); print("Figure 3 saved.")
print(f"\nPhase 1 complete. Figures -> {FIGURES_DIR}")

if not CFG.run_phase2:
    print("\nrun_phase2 = False. Stopping after the analytical audit.")
    sys.exit(0)


# =============================================================================
# CELL 6 — Tokenizer
# =============================================================================

print("\n" + "=" * 70)
print("CELL 6: Load Tokenizer")
print("=" * 70)

def load_tokenizer(cfg):
    for model_id in [cfg.model_name, cfg.model_name_fallback]:
        try:
            print(f"Trying tokenizer: {model_id}")
            tok = AutoTokenizer.from_pretrained(
                model_id, trust_remote_code=True, padding_side="left", use_fast=True)
            if tok.pad_token is None:
                tok.pad_token = tok.eos_token
                tok.pad_token_id = tok.eos_token_id
            print(f"Loaded: {model_id}  vocab={tok.vocab_size} "
                  f"eos={tok.eos_token!r} pad={tok.pad_token!r}")
            return tok, model_id
        except Exception as e:
            print(f"  failed: {e}")
    raise RuntimeError("Could not load any tokenizer.")

TOKENIZER, LOADED_MODEL_ID = load_tokenizer(CFG)
CFG.model_name = LOADED_MODEL_ID


# =============================================================================
# CELL 7 — Model (4-bit QLoRA)
# =============================================================================

print("\n" + "=" * 70)
print("CELL 7: Load Model (4-bit QLoRA)")
print("=" * 70)
print("""
LoRA-GRPO note: we do NOT hold a second reference model in memory.
Reference log-probs are obtained by DISABLING the LoRA adapters
(model.disable_adapter()), giving the frozen 4-bit base policy.
This is exact for the KL term and halves memory. Labelled "LoRA-GRPO".
""")

def load_model(cfg):
    bnb = BitsAndBytesConfig(
        load_in_4bit=cfg.load_in_4bit,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type=cfg.bnb_4bit_quant_type,
        bnb_4bit_use_double_quant=cfg.bnb_4bit_double_quant)
    print(f"Loading {cfg.model_name} ...")
    with memory_scope("model_load"):
        model = AutoModelForCausalLM.from_pretrained(
            cfg.model_name, quantization_config=bnb, device_map="auto",
            trust_remote_code=True, dtype=torch.bfloat16)
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False})
    model = get_peft_model(model, LoraConfig(
        r=cfg.lora_r, lora_alpha=cfg.lora_alpha,
        target_modules=cfg.lora_target_modules, lora_dropout=cfg.lora_dropout,
        bias="none", task_type=TaskType.CAUSAL_LM))
    model.print_trainable_parameters()
    return model

MODEL = load_model(CFG)
log_memory("after model load")


# =============================================================================
# CELL 8 — Prompt builder + safe generation
# =============================================================================

print("\n" + "=" * 70)
print("CELL 8: Prompt + Generation Helpers")
print("=" * 70)

COMPRESS_SYSTEM_PROMPT = (
    "You are a text compression agent. Compress the given text to "
    "approximately {target_tokens} tokens while preserving all key "
    "information, especially every number. Output ONLY the compressed text.")

def build_compression_prompt(text: str, target_tokens: int, tokenizer) -> str:
    messages = [
        {"role": "system",
         "content": COMPRESS_SYSTEM_PROMPT.format(target_tokens=target_tokens)},
        {"role": "user", "content": f"Compress this text:\n\n{text}"}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def safe_generate(model, tokenizer, prompt, max_new_tokens,
                  do_sample=False, temperature=0.9, top_p=0.95,
                  return_ids=False):
    """Bounded generation. Returns (text, n_new_tokens[, prompt_ids, comp_ids])."""
    enc = tokenizer(prompt, return_tensors="pt", truncation=True,
                    max_length=CFG.max_input_tokens, padding=False)
    input_ids = enc["input_ids"].to(model.device)
    attn = enc["attention_mask"].to(model.device)
    in_len = input_ids.shape[1]
    was_cache = model.config.use_cache
    model.config.use_cache = True
    try:
        out = model.generate(
            input_ids=input_ids, attention_mask=attn,
            generation_config=GenerationConfig(
                max_new_tokens=max_new_tokens, do_sample=do_sample,
                temperature=temperature if do_sample else 1.0,
                top_p=top_p if do_sample else 1.0,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id))
    finally:
        model.config.use_cache = was_cache
    new_ids = out[0, in_len:]
    text = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    if return_ids:
        return text, len(new_ids), input_ids[0], new_ids
    return text, len(new_ids)

_t = ("Janet's ducks lay 16 eggs per day. She eats 3 for breakfast and bakes "
      "muffins with 4. She sells the rest at $2 per egg. How much does she make?")
_n = len(TOKENIZER.encode(_t))
_g, _k = safe_generate(MODEL, TOKENIZER,
                       build_compression_prompt(_t, int(0.5*_n), TOKENIZER),
                       max_new_tokens=100)
print(f"Sanity: orig={_n} tok, comp={_k} tok, r_C={_k/_n:.3f}\n  {_g[:110]}")
cleanup_cuda()


# =============================================================================
# CELL 9-10 — Dataset load + preprocess
# =============================================================================

print("\n" + "=" * 70)
print("CELL 9-10: Dataset")
print("=" * 70)

def extract_gsm8k_answer(s: str) -> Optional[str]:
    m = re.search(r"####\s*([0-9,\-\.]+)", s)
    if m:
        return m.group(1).replace(",", "").strip()
    nums = re.findall(r"[-+]?\d+(?:\.\d+)?", s)
    return nums[-1] if nums else None

def extract_numbers(text: str) -> List[str]:
    """Numeric literals, normalised (strip commas, trailing .0)."""
    out = []
    for tok in re.findall(r"[-+]?\d[\d,]*(?:\.\d+)?", text):
        v = tok.replace(",", "")
        if v.endswith(".0"):
            v = v[:-2]
        out.append(v)
    return out

raw = load_dataset(CFG.dataset_name, CFG.dataset_config)
train_sh = raw["train"].shuffle(seed=CFG.seed)
test_sh = raw["test"].shuffle(seed=CFG.seed)
TRAIN_DS = train_sh.select(range(CFG.train_size))
EVAL_DS = test_sh.select(range(CFG.eval_size))
print(f"train={len(TRAIN_DS)}  eval={len(EVAL_DS)}")

def preprocess(ds, tokenizer, cfg, label):
    """
    r_T is drawn from U(0.1, 0.7) with a per-example deterministic seed, so
    BOTH training arms and BOTH eval passes see the identical r_T for the
    identical example. This is what makes the eval comparison PAIRED.
    """
    out, skipped = [], 0
    for idx, ex in enumerate(ds):
        q, a = ex["question"], ex["answer"]
        toks = tokenizer.encode(q, add_special_tokens=False)
        if len(toks) < 20:
            skipped += 1; continue
        if len(toks) > cfg.max_input_tokens - 60:
            toks = toks[:cfg.max_input_tokens - 60]
            q = tokenizer.decode(toks)
        orig_len = len(toks)
        rng = np.random.default_rng(cfg.seed * 100003 + idx)   # index-based, stable
        r_T = float(rng.uniform(cfg.r_T_min, cfg.r_T_max))
        tgt = max(5, int(r_T * orig_len))
        out.append({
            "idx": idx, "question": q, "gold_answer": extract_gsm8k_answer(a),
            "orig_numbers": extract_numbers(q),
            "prompt": build_compression_prompt(q, tgt, tokenizer),
            "orig_len": orig_len, "target_ratio": r_T, "target_tokens": tgt})
    lens = [x["orig_len"] for x in out]; rts = [x["target_ratio"] for x in out]
    print(f"{label}: {len(out)} kept, {skipped} skipped | "
          f"orig_len mean={np.mean(lens):.1f} | r_T mean={np.mean(rts):.3f} "
          f"[{min(rts):.3f}, {max(rts):.3f}]")
    return out

TRAIN_PROC = preprocess(TRAIN_DS, TOKENIZER, CFG, "train")
EVAL_PROC = preprocess(EVAL_DS, TOKENIZER, CFG, "eval")


# =============================================================================
# CELL 12b — Quality reward
# =============================================================================

print("\n" + "=" * 70)
print("CELL 12b: Quality Reward")
print("=" * 70)
print("""
IMPORTANT DEVIATION FROM A NAIVE IMPLEMENTATION
-----------------------------------------------
A tempting R_qual is "extract the answer from the compressed text and
exact-match it against the GSM8K gold answer". That is BROKEN here: we are
compressing the QUESTION, which contains the problem's numbers, not its
solution. Such a reward returns ~0 for nearly every rollout, R_qual becomes
constant, and group advantages end up driven entirely by R_len — destroying
the semantic counterweight the experiment depends on.

We use NUMBER RETENTION instead: the fraction of the original question's
numeric literals that survive in the compression. Deterministic, costs no
extra forward pass, and is semantically real for GSM8K — drop a number and
the problem becomes unsolvable. This is a documented simplification of
Cmprsr's summary-log-probability reward, not a replication of it.

Set cfg.use_solver_qual=True for the stronger (3x slower) variant that asks
the model to actually solve from the compressed text.
""")

def r_qual_number_retention(compression: str, orig_numbers: List[str]) -> float:
    if not orig_numbers:
        return 1.0
    comp = set(extract_numbers(compression))
    return sum(1.0 for n in orig_numbers if n in comp) / len(orig_numbers)

SOLVE_PROMPT = ("Solve the problem. Reply with ONLY the final number.\n\n{q}")

def r_qual_solver(model, tokenizer, compression: str, gold: str) -> float:
    if gold is None:
        return 0.0
    msgs = [{"role": "user", "content": SOLVE_PROMPT.format(q=compression)}]
    p = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    txt, _ = safe_generate(model, tokenizer, p, max_new_tokens=24, do_sample=False)
    nums = extract_numbers(txt)
    return float(bool(nums) and nums[-1] == gold)

def compute_reward(r_C, r_T, compression, ex, reward_type, cfg, model=None):
    if cfg.use_solver_qual and model is not None:
        rq = r_qual_solver(model, TOKENIZER, compression, ex["gold_answer"])
    else:
        rq = r_qual_number_retention(compression, ex["orig_numbers"])
    if reward_type == "original":
        rl = r_len_original(r_C, r_T)
    elif reward_type == "two_sided":
        rl = r_len_two_sided(r_C, r_T, cfg.lambda_two_sided)
    elif reward_type == "two_sided_strong":
        rl = r_len_two_sided(r_C, r_T, cfg.lambda_strong)
    else:
        raise ValueError(reward_type)
    assert math.isfinite(rl) and math.isfinite(rq)
    return {"r_qual": rq, "r_len": rl, "total": rq + rl,
            "r_C": r_C, "r_T": r_T, "delta_cr": delta_cr(r_C, r_T),
            "abs_delta": abs(delta_cr(r_C, r_T))}

_c = "Janet ducks lay 16 eggs. Eats 3, bakes 4. Sells rest at $2."
print(f"Retention demo: {r_qual_number_retention(_c, ['16','3','4','2']):.2f} "
      f"(all 4 kept)")
print(f"Retention demo: {r_qual_number_retention('Janet ducks lay eggs.', ['16','3','4','2']):.2f} "
      f"(all dropped)")


# =============================================================================
# CELL 16 — LoRA-GRPO training infrastructure
# =============================================================================

print("\n" + "=" * 70)
print("CELL 16: Training Infrastructure (LoRA-GRPO)")
print("=" * 70)

def sequence_logprobs(model, prompt_ids, comp_ids, use_adapter=True):
    """
    Per-token log-probs of comp_ids given prompt_ids.
    use_adapter=False -> frozen base policy (reference for KL).
    Returns tensor [L_comp].
    """
    ids = torch.cat([prompt_ids, comp_ids]).unsqueeze(0).to(model.device)
    if ids.shape[1] > CFG.max_seq_len:
        keep = CFG.max_seq_len - len(comp_ids)
        ids = torch.cat([prompt_ids[-keep:], comp_ids]).unsqueeze(0).to(model.device)
    ctx = model.disable_adapter() if not use_adapter else _null_ctx()
    with ctx:
        logits = model(input_ids=ids, attention_mask=torch.ones_like(ids)).logits
    L = len(comp_ids)
    logits = logits[0, -L-1:-1, :].float()
    return torch.log_softmax(logits, dim=-1).gather(
        1, comp_ids.unsqueeze(1).to(model.device)).squeeze(1)

@contextmanager
def _null_ctx():
    yield

def grpo_step(model, optimizer, scheduler, batch_examples, reward_type, cfg):
    """
    One optimiser step. For each example: sample G rollouts, compute rewards,
    group-relative advantages, standard-GRPO loss (1/L normalisation KEPT),
    plus a KL term against the adapter-disabled base policy.
    """
    optimizer.zero_grad(set_to_none=True)
    logs = []
    total_loss_val = 0.0
    n_backward = 0

    for ex in batch_examples:
        rollouts = []
        for _ in range(cfg.group_size):
            try:
                txt, n_new, p_ids, c_ids = safe_generate(
                    model, TOKENIZER, ex["prompt"], cfg.max_new_tokens,
                    do_sample=True, temperature=cfg.gen_temperature,
                    top_p=cfg.gen_top_p, return_ids=True)
            except torch.cuda.OutOfMemoryError:
                cleanup_cuda(); continue
            if n_new == 0:
                continue
            comp_len = max(1, len(TOKENIZER.encode(txt, add_special_tokens=False)))
            r_C = min(2.0, max(0.0, compute_compression_ratio(ex["orig_len"], comp_len)))
            rw = compute_reward(r_C, ex["target_ratio"], txt, ex, reward_type, cfg, model)
            rollouts.append({"p_ids": p_ids, "c_ids": c_ids, "text": txt, **rw})

        if len(rollouts) < 2:
            continue

        R = torch.tensor([r["total"] for r in rollouts], dtype=torch.float32)
        adv = (R - R.mean()) / (R.std(unbiased=False) + cfg.epsilon_denom)

        for i, r in enumerate(rollouts):
            try:
                lp = sequence_logprobs(model, r["p_ids"], r["c_ids"], use_adapter=True)
                with torch.no_grad():
                    lp_ref = sequence_logprobs(model, r["p_ids"], r["c_ids"],
                                               use_adapter=False)
                # standard GRPO: 1/L per-sequence normalisation retained
                pg = -(adv[i].to(lp.device) * lp).mean()
                kl = (lp - lp_ref).mean()
                loss = (pg + cfg.kl_coeff * kl) / (len(rollouts) * len(batch_examples))
                if not torch.isfinite(loss):
                    continue
                loss.backward()
                total_loss_val += loss.item(); n_backward += 1
            except torch.cuda.OutOfMemoryError:
                cleanup_cuda(); continue

        logs.extend([{k: v for k, v in r.items()
                      if k in ("r_qual", "r_len", "total", "r_C", "r_T",
                               "delta_cr", "abs_delta")} for r in rollouts])

    if n_backward > 0:
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], cfg.max_grad_norm)
        optimizer.step(); scheduler.step()
    optimizer.zero_grad(set_to_none=True)
    cleanup_cuda()
    return total_loss_val, logs


def train_arm(model, arm_name, reward_type, cfg, train_data):
    print(f"\n{'='*70}\nTRAIN ARM: {arm_name}  (reward_type={reward_type})\n{'='*70}")
    trainable = [p for p in model.parameters() if p.requires_grad]
    for p in trainable:
        p.data = p.data.float()
    opt = torch.optim.AdamW(trainable, lr=cfg.learning_rate,
                            weight_decay=cfg.weight_decay)
    sched = get_linear_schedule_with_warmup(opt, cfg.warmup_steps, cfg.num_steps)

    history, t0 = [], time.time()
    per_step = max(1, cfg.grad_accum_steps // cfg.group_size)
    for step in range(cfg.num_steps):
        lo = (step * per_step) % len(train_data)
        batch = [train_data[(lo + j) % len(train_data)] for j in range(per_step)]
        loss, logs = grpo_step(model, opt, sched, batch, reward_type, cfg)
        if logs:
            rec = {
                "step": step, "loss": loss,
                "mean_r_C": float(np.mean([l["r_C"] for l in logs])),
                "mean_delta": float(np.mean([l["delta_cr"] for l in logs])),
                "mean_abs_delta": float(np.mean([l["abs_delta"] for l in logs])),
                "std_delta": float(np.std([l["delta_cr"] for l in logs])),
                "mean_r_qual": float(np.mean([l["r_qual"] for l in logs])),
                "mean_r_len": float(np.mean([l["r_len"] for l in logs])),
                "frac_below_target": float(np.mean(
                    [1.0 if l["delta_cr"] < 0 else 0.0 for l in logs])),
            }
            history.append(rec)
            if step % 5 == 0 or step == cfg.num_steps - 1:
                print(f"  step {step:3d} | loss {loss:+.4f} | "
                      f"Δ_CR {rec['mean_delta']:+.3f} "
                      f"(|Δ| {rec['mean_abs_delta']:.3f}, σ {rec['std_delta']:.3f}) | "
                      f"below-target {rec['frac_below_target']*100:4.0f}% | "
                      f"R_qual {rec['mean_r_qual']:.2f} | "
                      f"{(time.time()-t0)/60:.1f}m")
    print(f"Arm '{arm_name}' finished in {(time.time()-t0)/60:.1f} min")
    return history


# =============================================================================
# CELL 17-19 — Run both arms + PAIRED evaluation
# =============================================================================

@torch.no_grad()
def evaluate(model, eval_data, cfg, tag):
    """
    Greedy decoding, identical examples and identical r_T for every arm
    -> results are PAIRED and can be compared per-example.
    """
    print(f"\nEvaluating [{tag}] on {len(eval_data)} held-out examples (greedy)...")
    rows = []
    for ex in eval_data:
        try:
            txt, _ = safe_generate(model, TOKENIZER, ex["prompt"],
                                   cfg.max_new_tokens, do_sample=False)
        except torch.cuda.OutOfMemoryError:
            cleanup_cuda(); continue
        if not txt:
            txt = "[EMPTY]"
        comp_len = max(1, len(TOKENIZER.encode(txt, add_special_tokens=False)))
        r_C = min(2.0, compute_compression_ratio(ex["orig_len"], comp_len))
        rows.append({
            "idx": ex["idx"], "r_T": ex["target_ratio"], "r_C": r_C,
            "delta_cr": delta_cr(r_C, ex["target_ratio"]),
            "abs_delta": abs(delta_cr(r_C, ex["target_ratio"])),
            "r_qual": r_qual_number_retention(txt, ex["orig_numbers"]),
            "comp_len": comp_len, "text": txt})
    cleanup_cuda()
    return rows

def save_adapter(model, name):
    p = OUTPUT_DIR / f"adapter_{name}"
    model.save_pretrained(str(p)); return p

def reset_adapter(model):
    """Re-initialise LoRA weights to zero-effect so arm 2 starts from base."""
    for n, p in model.named_parameters():
        if "lora_A" in n:
            torch.nn.init.kaiming_uniform_(p, a=math.sqrt(5))
        elif "lora_B" in n:
            torch.nn.init.zeros_(p)
    print("LoRA adapters reset to base-equivalent initialisation.")

RESULTS = {}

# ---- Arm 1: baseline (original one-sided reward) ----
set_seed(CFG.seed)
reset_adapter(MODEL)
RESULTS["baseline_history"] = train_arm(MODEL, "baseline", "original", CFG, TRAIN_PROC)
RESULTS["baseline_eval"] = evaluate(MODEL, EVAL_PROC, CFG, "baseline")
save_adapter(MODEL, "baseline")

# ---- Arm 2: proposed (additive two-sided reward) ----
set_seed(CFG.seed)
reset_adapter(MODEL)
RESULTS["proposed_history"] = train_arm(MODEL, "proposed", "two_sided", CFG, TRAIN_PROC)
RESULTS["proposed_eval"] = evaluate(MODEL, EVAL_PROC, CFG, "proposed")
save_adapter(MODEL, "proposed")


# =============================================================================
# CELL 20 — Metrics: PAIRED comparison + full distribution
# =============================================================================

print("\n" + "=" * 70)
print("CELL 20: Results  [EMPIRICAL]")
print("=" * 70)

from scipy import stats

b = {r["idx"]: r for r in RESULTS["baseline_eval"]}
p = {r["idx"]: r for r in RESULTS["proposed_eval"]}
common = sorted(set(b) & set(p))
print(f"Paired on {len(common)} examples present in both arms.")

bd = np.array([b[i]["delta_cr"] for i in common])
pd_ = np.array([p[i]["delta_cr"] for i in common])
ba = np.abs(bd); pa = np.abs(pd_)
bq = np.array([b[i]["r_qual"] for i in common])
pq = np.array([p[i]["r_qual"] for i in common])

def describe(name, d, a, q):
    print(f"\n{name}")
    print(f"  Δ_CR    mean {d.mean():+.4f}  σ {d.std():.4f}  "
          f"median {np.median(d):+.4f}")
    print(f"  |Δ_CR|  mean {a.mean():.4f}   σ {a.std():.4f}")
    print(f"  quantiles Δ_CR: p10 {np.percentile(d,10):+.3f}  "
          f"p25 {np.percentile(d,25):+.3f}  p75 {np.percentile(d,75):+.3f}  "
          f"p90 {np.percentile(d,90):+.3f}")
    print(f"  below target (Δ<0): {100*np.mean(d<0):.0f}%   "
          f"R_qual mean {q.mean():.3f}")

describe("BASELINE (original one-sided R_len)", bd, ba, bq)
describe("PROPOSED (additive two-sided R_len)", pd_, pa, pq)

print("\n--- PAIRED tests (same examples, same r_T) ---")
t_abs, pv_abs = stats.ttest_rel(ba, pa)
try:
    w_abs, wp_abs = stats.wilcoxon(ba, pa)
except ValueError:
    w_abs, wp_abs = float("nan"), float("nan")
t_q, pv_q = stats.ttest_rel(bq, pq)
diff = ba - pa
d_eff = diff.mean() / (diff.std(ddof=1) + 1e-12)

print(f"  |Δ_CR| paired t-test : t={t_abs:+.3f}  p={pv_abs:.4f}")
print(f"  |Δ_CR| Wilcoxon      : W={w_abs:.1f}  p={wp_abs:.4f}")
print(f"  |Δ_CR| Cohen's d_z   : {d_eff:+.3f}")
print(f"  R_qual paired t-test : t={t_q:+.3f}  p={pv_q:.4f}  "
      f"(checking the proposal did NOT cost semantic retention)")
print(f"  σ(Δ_CR) baseline {bd.std():.4f} -> proposed {pd_.std():.4f} "
      f"({100*(pd_.std()-bd.std())/(bd.std()+1e-12):+.1f}%)")

ALPHA = 0.05
print("\n--- VERDICT ---")
if pv_abs < ALPHA and pa.mean() < ba.mean():
    print(f"  Proposed reward SIGNIFICANTLY tightened |Δ_CR| (p={pv_abs:.4f}).")
elif pv_abs < ALPHA:
    print(f"  Significant difference, but in the WRONG direction (p={pv_abs:.4f}).")
else:
    print(f"  NO significant difference (p={pv_abs:.4f}, n={len(common)}).")
    print("  Honest reading: the mechanism is present analytically (Phase 1),")
    print(f"  but {CFG.num_steps} steps on {CFG.train_size} examples did not")
    print("  resolve an effect. This is a legitimate negative result — report")
    print("  it as underpowered rather than claiming a noisy delta.")

RESULTS["metrics"] = {
    "n_paired": len(common),
    "baseline": {"mean_delta": float(bd.mean()), "std_delta": float(bd.std()),
                 "mean_abs_delta": float(ba.mean()), "mean_r_qual": float(bq.mean())},
    "proposed": {"mean_delta": float(pd_.mean()), "std_delta": float(pd_.std()),
                 "mean_abs_delta": float(pa.mean()), "mean_r_qual": float(pq.mean())},
    "paired_t_abs_delta": {"t": float(t_abs), "p": float(pv_abs)},
    "wilcoxon_abs_delta": {"W": float(w_abs), "p": float(wp_abs)},
    "cohens_dz": float(d_eff),
    "paired_t_r_qual": {"t": float(t_q), "p": float(pv_q)},
}


# =============================================================================
# CELL 21 — Figure 4 (empirical)
# =============================================================================

fig4, ax4 = plt.subplots(2, 2, figsize=(13, 9))

# (0,0) training Δ_CR trajectory
for hist, lab, c in [(RESULTS["baseline_history"], "baseline (one-sided)", "tab:blue"),
                     (RESULTS["proposed_history"], "proposed (two-sided)", "tab:green")]:
    if hist:
        ax4[0,0].plot([h["step"] for h in hist], [h["mean_delta"] for h in hist],
                      color=c, lw=1.8, label=lab)
ax4[0,0].axhline(0, color="k", ls="--", lw=1)
ax4[0,0].set_xlabel("optimiser step"); ax4[0,0].set_ylabel(r"mean $\Delta_{CR}$")
ax4[0,0].set_title("Training: mean $\\Delta_{CR}$ [Empirical]"); ax4[0,0].legend(fontsize=8)

# (0,1) training |Δ_CR|
for hist, lab, c in [(RESULTS["baseline_history"], "baseline", "tab:blue"),
                     (RESULTS["proposed_history"], "proposed", "tab:green")]:
    if hist:
        ax4[0,1].plot([h["step"] for h in hist], [h["mean_abs_delta"] for h in hist],
                      color=c, lw=1.8, label=lab)
ax4[0,1].set_xlabel("optimiser step"); ax4[0,1].set_ylabel(r"mean $|\Delta_{CR}|$")
ax4[0,1].set_title("Training: adherence error [Empirical]"); ax4[0,1].legend(fontsize=8)

# (1,0) eval Δ_CR distribution — the headline
parts = ax4[1,0].violinplot([bd, pd_], showmeans=True, showextrema=True)
ax4[1,0].axhline(0, color="k", ls="--", lw=1)
ax4[1,0].set_xticks([1,2]); ax4[1,0].set_xticklabels(["baseline", "proposed"])
ax4[1,0].set_ylabel(r"$\Delta_{CR}$ (held-out, greedy)")
ax4[1,0].set_title(f"Eval $\\Delta_{{CR}}$ distribution, paired n={len(common)}\n"
                   f"σ: {bd.std():.3f} → {pd_.std():.3f}  (p={pv_abs:.3f})")

# (1,1) per-example paired change
ax4[1,1].scatter(ba, pa, alpha=0.7, color="tab:purple")
lim = max(ba.max(), pa.max()) * 1.1 if len(ba) else 1
ax4[1,1].plot([0, lim], [0, lim], "k--", lw=1)
ax4[1,1].set_xlim(0, lim); ax4[1,1].set_ylim(0, lim)
ax4[1,1].set_xlabel(r"baseline $|\Delta_{CR}|$")
ax4[1,1].set_ylabel(r"proposed $|\Delta_{CR}|$")
ax4[1,1].set_title("Paired per-example adherence error\n(below diagonal = proposed better)")

fig4.suptitle("Figure 4 [Empirical]: LoRA-GRPO, one-sided vs additive two-sided reward",
              fontsize=13)
fig4.tight_layout()
fig4.savefig(FIGURES_DIR / "figure4_training_results.png", bbox_inches="tight")
plt.close(fig4)
print("Figure 4 saved.")


# =============================================================================
# CELL 22-23 — Save artifacts + interpretation
# =============================================================================

with open(OUTPUT_DIR / "results.json", "w") as f:
    json.dump({k: v for k, v in RESULTS.items() if k != "raw"}, f,
              indent=2, default=float)

print("\n" + "=" * 70)
print("INTERPRETATION")
print("=" * 70)
print(f"""
Phase 1 (analytical, Figures 1-3) — reproducible from the paper's equations
alone: Cmprsr's R_len is flat at 1.0 across the whole interval r_C in
[0, r_T], a region exactly r_T wide. Standard GRPO's 1/L normalisation gives
the LARGEST per-token gradient weight precisely inside that interval (2x at
half the target length, 4x at a quarter).

Phase 2 (empirical, Figure 4) — LoRA-GRPO on {CFG.train_size} GSM8K examples,
{CFG.num_steps} steps, G={CFG.group_size}, paired eval on {len(common)} held-out
examples with identical r_T per example across arms.
  baseline |Δ_CR| = {ba.mean():.4f}   proposed |Δ_CR| = {pa.mean():.4f}
  paired p = {pv_abs:.4f}, Cohen's d_z = {d_eff:+.3f}
  R_qual  {bq.mean():.3f} -> {pq.mean():.3f}  (p={pv_q:.4f})

Scope limits to state out loud:
  - Not a Cmprsr replication: LoRA not full fine-tuning, number-retention
    not summary-log-prob reward, GSM8K only, no MeetingBank/LongBench.
  - Both arms use standard GRPO; the reward is the only variable. A Dr. GRPO
    arm would test the optimiser half of the mechanism and is not run here.
  - n={len(common)} and {CFG.num_steps} steps is small. Treat a null result as
    underpowered, not as refutation.
""")
print(f"Artifacts -> {OUTPUT_DIR}")

CELL 1: Environment Diagnostics
PyTorch version     : 2.10.0+cu128
GPU name            : Tesla T4
Total VRAM          : 14.56 GB
CUDA version        : 12.8
GPU check: OK (>=14 GB VRAM)
transformers        : 5.0.0
peft                : 0.19.1
trl                 : not installed yet
bitsandbytes        : not installed yet

CELL 2: Installing dependencies
Installation complete.

CELL 3: Imports
All imports successful.

CELL 4: Reproducibility + Memory Utilities
Seeds set to 42
Output directory: /kaggle/working/cmprsr_research
[MEM after cell 4] alloc=0.00GB reserved=0.00GB total=14.56GB

CELL 5: Configuration

Final configuration:
  seed                      : 42
  model_name                : Qwen/Qwen3-4B-Instruct-2507
  model_name_fallback       : Qwen/Qwen3-4B
  load_in_4bit              : True
  bnb_4bit_quant_type       : nf4
  bnb_4bit_double_quant     : True
  lora_r                    : 16
  lora_alpha                : 32
  lora_dropout              : 0.05
  lora_target_modules   

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

train=80  eval=30
train: 80 kept, 0 skipped | orig_len mean=61.2 | r_T mean=0.389 [0.102, 0.699]
eval: 30 kept, 0 skipped | orig_len mean=62.0 | r_T mean=0.437 [0.116, 0.699]

CELL 12b: Quality Reward

IMPORTANT DEVIATION FROM A NAIVE IMPLEMENTATION
-----------------------------------------------
A tempting R_qual is "extract the answer from the compressed text and
exact-match it against the GSM8K gold answer". That is BROKEN here: we are
compressing the QUESTION, which contains the problem's numbers, not its
solution. Such a reward returns ~0 for nearly every rollout, R_qual becomes
constant, and group advantages end up driven entirely by R_len — destroying
the semantic counterweight the experiment depends on.

We use NUMBER RETENTION instead: the fraction of the original question's
numeric literals that survive in the compression. Deterministic, costs no
extra forward pass, and is semantically real for GSM8K — drop a number and
the problem becomes unsolvable. This is a documented sim